In [ ]:
%sql
USE workspace.taco_bueno;

## 0. Data quality snapshot

In [ ]:
%sql
SELECT
  SUM(rows_loaded)               AS rows_loaded,
  SUM(orders)                    AS orders,
  SUM(missing_hour)              AS missing_hour,
  SUM(missing_store)             AS missing_store,
  SUM(missing_channel)           AS missing_channel,
  SUM(discount_not_numeric)      AS discount_not_numeric,
  SUM(discount_above_price)      AS discount_above_price,
  SUM(possible_duplicate_lines)  AS possible_duplicate_lines,
  ROUND(SUM(computed_revenue_usd), 2) AS computed_revenue_usd,
  ROUND(SUM(raw_total_column_sum), 2) AS raw_total_column_sum,
  ROUND(SUM(raw_total_column_sum) / SUM(computed_revenue_usd), 2) AS raw_vs_computed_ratio
FROM workspace.taco_bueno.mv_data_quality_summary;

## Analysis A: Top Revenue & Best-Selling Menu Items

In [ ]:
%sql
SELECT menu_item, total_units, gross_revenue_usd, discount_usd, net_revenue_usd, orders
FROM workspace.taco_bueno.mv_top_menu_items
ORDER BY net_revenue_usd DESC
LIMIT 10;

## Analysis B: Promotion Effectiveness (Discounted vs. Regular Price)

In [ ]:
%sql
SELECT * FROM workspace.taco_bueno.mv_promo_vs_regular;

## Analysis C: Sales Channel Performance

In [ ]:
%sql
SELECT sales_channel, orders, revenue_usd, avg_ticket_usd, discount_usd, pct_orders_with_promo,
       ROUND(100 * revenue_usd / SUM(revenue_usd) OVER (), 2) AS pct_of_revenue
FROM workspace.taco_bueno.mv_channel_performance
ORDER BY revenue_usd DESC;

## Analysis D: The "Lift" Test – Do Promos Increase Ticket Size?

In [ ]:
%sql
SELECT
  CASE WHEN has_promo THEN 'With promo' ELSE 'Without promo' END AS ticket_type,
  COUNT(*)                                        AS orders,
  ROUND(SUM(basket_total_usd), 2)                 AS revenue_usd,
  ROUND(AVG(basket_total_usd), 2)                 AS avg_basket_usd,
  ROUND(PERCENTILE_APPROX(basket_total_usd, 0.5), 2) AS median_basket_usd,
  ROUND(AVG(units), 2)                            AS avg_units
FROM workspace.taco_bueno.mv_order_summary
GROUP BY has_promo;

In [ ]:
%sql
SELECT * FROM workspace.taco_bueno.mv_promo_lift ORDER BY sales_channel, ticket_type;

## Analysis E: Cannibalization – Discount Dependency

In [ ]:
%sql
SELECT menu_item, total_units, discounted_units, full_price_units, discount_dependency_pct, revenue_usd
FROM workspace.taco_bueno.mv_discount_dependency
WHERE total_units > 1000
ORDER BY discount_dependency_pct DESC
LIMIT 15;

## Analysis F: Seasonality & Peak Hours

In [ ]:
%sql
SELECT
  order_hour,
  CASE WHEN order_hour = 0 THEN '12 AM' WHEN order_hour < 12 THEN CONCAT(order_hour, ' AM')
       WHEN order_hour = 12 THEN '12 PM' ELSE CONCAT(order_hour - 12, ' PM') END AS hour_label,
  SUM(orders)                    AS orders,
  ROUND(SUM(revenue_usd), 2)     AS revenue_usd,
  ROUND(SUM(revenue_usd) / SUM(orders), 2) AS avg_ticket_usd,
  ROUND(SUM(pct_orders_with_promo * orders) / SUM(orders), 2) AS pct_orders_with_promo
FROM workspace.taco_bueno.mv_hourly_seasonality
WHERE order_hour IS NOT NULL
GROUP BY order_hour
ORDER BY order_hour;

### Day of week

In [ ]:
%sql
SELECT day_num, day_of_week, SUM(orders) AS orders, ROUND(SUM(revenue_usd), 2) AS revenue_usd,
       ROUND(SUM(revenue_usd) / SUM(orders), 2) AS avg_ticket_usd
FROM workspace.taco_bueno.mv_hourly_seasonality
WHERE day_of_week IS NOT NULL
GROUP BY day_num, day_of_week
ORDER BY day_num;

## Analysis G: Halo Effect / Attach Rate
What customers add to the order when they buy a discounted item (top 5 discounted items by orders).

In [ ]:
%sql
WITH top_promo AS (
  SELECT menu_item FROM workspace.taco_bueno.mv_discount_dependency
  ORDER BY discounted_units DESC LIMIT 5
)
SELECT h.promo_item, h.rank_num, h.companion_item, h.orders_together, h.companion_revenue_usd
FROM workspace.taco_bueno.mv_halo_effect h
JOIN top_promo t ON h.promo_item = t.menu_item
WHERE h.rank_num <= 3
ORDER BY h.promo_item, h.rank_num;

## Analysis H: Promo ROI / Discount Efficiency

In [ ]:
%sql
SELECT menu_item, total_units, revenue_usd, discount_usd, discount_dependency_pct,
       revenue_per_discount_dollar, recommendation
FROM workspace.taco_bueno.mv_promo_roi
WHERE discount_usd > 0
ORDER BY revenue_per_discount_dollar ASC;

## Analysis I: Promos that should COME BACK

In [ ]:
%sql
WITH m AS (
  SELECT menu_item, month, units,
         CASE WHEN promo_units >= 0.2 * units THEN 1 ELSE 0 END AS promo_month
  FROM workspace.taco_bueno.mv_monthly_item_trend
  WHERE month IS NOT NULL
),
s AS (
  SELECT menu_item,
         SUM(promo_month)                                  AS promo_months,
         SUM(1 - promo_month)                              AS non_promo_months,
         AVG(CASE WHEN promo_month = 1 THEN units END)     AS avg_units_promo_months,
         AVG(CASE WHEN promo_month = 0 THEN units END)     AS avg_units_non_promo_months
  FROM m GROUP BY menu_item
)
SELECT *,
       ROUND(100 * (avg_units_promo_months / NULLIF(avg_units_non_promo_months, 0) - 1), 1) AS volume_uplift_pct
FROM s
WHERE promo_months >= 1 AND non_promo_months >= 1 AND avg_units_non_promo_months >= 100
ORDER BY volume_uplift_pct DESC
LIMIT 15;

In [ ]:
%sql
SELECT month, units, promo_units, revenue_usd
FROM workspace.taco_bueno.mv_monthly_item_trend
WHERE menu_item = 'Steak Soft Taco'   -- change the item
ORDER BY month;

## Final recommendation: LIVE / DIE / COME BACK

In [ ]:
%sql
WITH m AS (
  SELECT menu_item, units, CASE WHEN promo_units >= 0.2 * units THEN 1 ELSE 0 END AS promo_month
  FROM workspace.taco_bueno.mv_monthly_item_trend WHERE month IS NOT NULL
),
uplift AS (
  SELECT menu_item,
         ROUND(100 * (AVG(CASE WHEN promo_month = 1 THEN units END)
                    / NULLIF(AVG(CASE WHEN promo_month = 0 THEN units END), 0) - 1), 1) AS volume_uplift_pct,
         SUM(CASE WHEN promo_month = 1 THEN 1 ELSE 0 END) AS promo_months
  FROM m GROUP BY menu_item
)
SELECT
  r.menu_item, r.revenue_usd, r.discount_usd, r.revenue_per_discount_dollar,
  r.discount_dependency_pct, u.volume_uplift_pct,
  CASE
    WHEN r.revenue_per_discount_dollar < 6 OR r.discount_dependency_pct >= 70 THEN 'DIE'
    WHEN u.volume_uplift_pct >= 25 AND r.discount_dependency_pct < 50       THEN 'COME BACK'
    WHEN r.revenue_per_discount_dollar >= 10 AND r.discount_dependency_pct < 40 THEN 'LIVE'
    ELSE 'REVIEW'
  END AS decision
FROM workspace.taco_bueno.mv_promo_roi r
LEFT JOIN uplift u ON r.menu_item = u.menu_item
WHERE r.discount_usd > 0
ORDER BY decision, r.revenue_usd DESC;